# Battery RUL Classification — Inference (V2)

Sliding-window inference over whole cell lives, using the V2 pipeline.

What changed from `predict_clf_bml.ipynb`:

| | old | V2 |
|---|---|---|
| dataset | `dataset_clf_bml` | `dataset_clf_bml_v2` |
| model class | `train_clf_bml` (pulled in the whole MIT pipeline) | `model_clf` |
| architecture | retyped by hand (`SUMMARY_FEATS = 12`, `gru_layers=2`) | read from `model_config.json` |
| class names | hard-coded, last one wrong (`RUL<100` vs actual `<=`) | derived from `RUL_EDGES` |
| per-cell cache | rebuilt for every window | built once per cell |
| confusion matrix | three near-identical cells (77–88% duplicate) | one function |

## 1 — Imports and configuration

In [ ]:
import json
import os

import joblib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import colormaps
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix

from dataset_clf_bml_v2 import (CellCache, N_CLASSES, N_EARLY, N_INPUT,
                                N_RANDOM, REF_CYCLE, RUL_EDGES, V_BINS,
                                build_sample_tensors, load_all_npz,
                                rul_to_class, window_rul)
from model_clf import BatteryRULClassifier

# ── Config ──────────────────────────────────────────────────────────────
dataset      = "MATR"
DATA_DIR     = f"./content_bml/{dataset}"
CKPT_DIR     = f"./checkpoints_clf_bml_{dataset}/"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# Which cells to analyse. None = every cell found in DATA_DIR. To reproduce
# a specific split, paste the TestCellName list the trainer printed.
TEST_CELLS   = None

print(f"Device: {DEVICE}")


def class_names(edges=RUL_EDGES):
    """Readable name per class, derived from the dataset's own thresholds.

    The old notebook hard-coded this list and its last entry read "RUL<100"
    while rul_to_class actually uses <= 100, so a cell with exactly 100
    cycles left was mislabelled in every plot legend.
    """
    names = [f"RUL>{edges[0]}"]
    for i in range(len(edges) - 1):
        names.append(f"{edges[i + 1]}<RUL<={edges[i]}")
    names.append(f"RUL<={edges[-1]}")
    return names


CLASS_NAMES  = class_names()
CLASS_SHORT  = [f"C{i}" for i in range(N_CLASSES)]
CLASS_COLORS = ["#1565C0", "#2E7D32", "#F9A825", "#E65100", "#B71C1C"]
print("Classes:", CLASS_NAMES)

## 2 — Load data, model and scalers

In [ ]:
cells = load_all_npz(DATA_DIR)
print(f"Loaded {len(cells)} cells")

In [ ]:
# Architecture comes from the file the trainer wrote next to the weights,
# so it can never drift from what the checkpoint actually contains. The old
# notebook retyped SUMMARY_FEATS and gru_layers by hand -- a silent
# shape-mismatch waiting to happen after any --cnn_dim / --gru_dim run.
with open(os.path.join(CKPT_DIR, "model_config.json"), encoding="utf-8") as f:
    cfg = json.load(f)
print("model_config.json:", json.dumps(cfg, indent=2))

model = BatteryRULClassifier(
    cnn_dim=cfg["cnn_dim"],
    gru_dim=cfg["gru_dim"],
    gru_layers=cfg["gru_layers"],
    summary_feats=cfg["summary_feats"],
    n_classes=cfg["n_classes"],
    dropout=cfg["dropout"],
).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "best_clf_bml.pt"),
                                 map_location=DEVICE, weights_only=True))
model.eval()

scalers = (joblib.load(os.path.join(CKPT_DIR, "dq_scaler_bml.pkl")),
           joblib.load(os.path.join(CKPT_DIR, "summary_scaler_bml.pkl")))

print(f"Model loaded - {sum(p.numel() for p in model.parameters()):,} parameters")

## 3 — Sliding-window inference

In [ ]:
@torch.no_grad()
def predict_cell(cell, scalers, model=model):
    """Slide the input window across a cell's whole life.

    One CellCache is built per cell and reused for every window. This is
    the correct usage rather than a speed trick: measured at batch size 1,
    it makes no visible difference (0.36s vs 0.37s over 200 windows)
    because the GPU forward pass dominates, not the feature assembly.

    Args:
        cell: A cell dict from load_all_npz.
        scalers: (dq_scaler, summary_scaler) from the checkpoint directory.
        model: The classifier to run.

    Returns:
        Dict of arrays aligned to the window's END cycle, plus cycle_life
        and the sliding-window accuracy.
    """
    cycle_life  = int(cell["cycle_life"])
    cycle_index = cell["cycle_index"]
    n_cyc       = cycle_index.size
    cache       = CellCache(cell)
    dq_sc, sum_sc = scalers

    end_cycles, pred_classes, true_classes, pred_probs = [], [], [], []

    for start in range(N_EARLY, n_cyc - N_RANDOM + 1):
        dq_seq, sum_seq = build_sample_tensors(cell, start, dq_sc, sum_sc,
                                               cache=cache)
        logits = model(dq_seq.unsqueeze(0).to(DEVICE),
                       sum_seq.unsqueeze(0).to(DEVICE)).squeeze(0)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()

        # window_rul and rul_to_class come from the dataset module, so the
        # label a window gets here is the same one it would get in training.
        rul = window_rul(cycle_life, cycle_index, start)

        end_cycles.append(int(cycle_index[start + N_RANDOM - 1]))
        pred_classes.append(int(probs.argmax()))
        true_classes.append(rul_to_class(rul))
        pred_probs.append(probs)

    pred_classes = np.array(pred_classes)
    true_classes = np.array(true_classes)
    return {
        "end_cycles":   np.array(end_cycles),
        "pred_classes": pred_classes,
        "true_classes": true_classes,
        "pred_probs":   np.array(pred_probs),
        "cycle_life":   cycle_life,
        "accuracy":     float((pred_classes == true_classes).mean()),
    }

## 4 — Single cell: predicted vs ground truth

In [ ]:
CELL_ID = sorted(cells)[0] if TEST_CELLS is None else TEST_CELLS[0]
print(f"Cell: {CELL_ID}")

res = predict_cell(cells[CELL_ID], scalers)
print(f"Sliding-window accuracy: {res['accuracy']:.4f}  "
      f"({len(res['end_cycles'])} windows, cycle_life={res['cycle_life']})")

fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                         gridspec_kw={"height_ratios": [2, 2, 1.2]})
fig.suptitle(f"RUL Classification - {CELL_ID}  "
             f"(cycle life {res['cycle_life']}, accuracy {res['accuracy']:.2%})",
             fontsize=13, fontweight="bold")

cycles, pred_cls = res["end_cycles"], res["pred_classes"]
true_cls, probs  = res["true_classes"], res["pred_probs"]
correct          = pred_cls == true_cls

ax = axes[0]
ax.step(cycles, true_cls, where="post", color="#1565C0", lw=2.5,
        label="True class", zorder=3)
ax.step(cycles, pred_cls, where="post", color="#E53935", lw=1.8, ls="--",
        label="Predicted class", zorder=4)
ax.fill_between(cycles, -0.4, N_CLASSES - 0.6, where=correct,
                alpha=0.08, color="green", step="post")
ax.fill_between(cycles, -0.4, N_CLASSES - 0.6, where=~correct,
                alpha=0.12, color="red", step="post")
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_ylim(-0.4, N_CLASSES - 0.6)
ax.set_ylabel("RUL Class"); ax.set_title("Predicted vs True Class Over Cycle Life")
ax.legend(fontsize=9); ax.grid(True, alpha=0.25)

ax = axes[1]
for c in range(N_CLASSES):
    ax.plot(cycles, probs[:, c], color=CLASS_COLORS[c], lw=1.5,
            label=CLASS_NAMES[c], alpha=0.85)
ax.set_ylabel("Softmax Probability"); ax.set_ylim(-0.05, 1.05)
ax.set_title("Predicted Class Probabilities")
ax.legend(fontsize=8, ncol=N_CLASSES, loc="upper right"); ax.grid(True, alpha=0.25)

ax = axes[2]
error = pred_cls - true_cls
ax.bar(cycles, error, width=1.0,
       color=["#E53935" if e else "#43A047" for e in error], alpha=0.75)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Window End Cycle"); ax.set_ylabel("Class Error\n(pred - true)")
ax.set_title("Classification Error per Window"); ax.grid(True, alpha=0.25)
ax.legend(handles=[mpatches.Patch(color="#43A047", alpha=0.75, label="Correct"),
                   mpatches.Patch(color="#E53935", alpha=0.75, label="Incorrect")],
          fontsize=9)

plt.tight_layout()
plt.savefig(f"{CELL_ID}_clf_prediction_V2.png", dpi=150, bbox_inches="tight")
plt.show()

## 5 — All cells

In [ ]:
target_cells = sorted(cells) if TEST_CELLS is None else TEST_CELLS
missing = [c for c in target_cells if c not in cells]
if missing:
    print(f"WARNING: {len(missing)} cells not found in {DATA_DIR}: {missing[:5]}")
target_cells = [c for c in target_cells if c in cells]

print(f"Running inference on {len(target_cells)} cells...")
all_results = {}
for cid in target_cells:
    r = predict_cell(cells[cid], scalers)
    if len(r["end_cycles"]):
        all_results[cid] = r
print(f"Done: {len(all_results)} cells")

n, ncols = len(all_results), 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3),
                         constrained_layout=True, squeeze=False)
fig.suptitle("All Cells - Predicted vs True RUL Class",
             fontsize=14, fontweight="bold", y=1.01)
axes_flat = axes.flatten()

for ax, (cid, r) in zip(axes_flat, all_results.items()):
    cyc, p, t = r["end_cycles"], r["pred_classes"], r["true_classes"]
    ok = p == t
    ax.step(cyc, t, where="post", color="#1565C0", lw=1.8)
    ax.step(cyc, p, where="post", color="#E53935", lw=1.4, ls="--")
    ax.fill_between(cyc, -0.4, N_CLASSES - 0.6, where=ok,  alpha=0.08, color="green", step="post")
    ax.fill_between(cyc, -0.4, N_CLASSES - 0.6, where=~ok, alpha=0.14, color="red",   step="post")
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_SHORT, fontsize=6)
    ax.set_ylim(-0.4, N_CLASSES - 0.6)
    ax.set_title(f"{cid}\nacc={r['accuracy']:.2%}", fontsize=7.5)
    ax.tick_params(labelsize=6); ax.grid(True, alpha=0.2)

for ax in axes_flat[len(all_results):]:
    ax.set_visible(False)

fig.legend(handles=[
    Line2D([0], [0], color="#1565C0", lw=1.8, label="True class"),
    Line2D([0], [0], color="#E53935", lw=1.4, ls="--", label="Pred class"),
    mpatches.Patch(color="green", alpha=0.2, label="Correct"),
    mpatches.Patch(color="red", alpha=0.25, label="Incorrect")],
    loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.savefig(f"all_cells_clf_V2_{dataset}.png", dpi=130, bbox_inches="tight")
plt.show()

accs = [r["accuracy"] for r in all_results.values()]
print(f"\nAll-cell accuracy  mean={np.mean(accs):.4f}  std={np.std(accs):.4f}  "
      f"min={np.min(accs):.4f}  max={np.max(accs):.4f}")

## 6 — Overlay, coloured by cycle life

In [ ]:
lives = [r["cycle_life"] for r in all_results.values()]
vmin, vmax = min(lives), max(lives)
cmap = colormaps.get_cmap("plasma")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("All Cells Overlay - Coloured by Cycle Life",
             fontsize=13, fontweight="bold")

for ax, key in zip(axes, ["true_classes", "pred_classes"]):
    for r in all_results.values():
        ax.step(r["end_cycles"], r[key], where="post",
                color=cmap((r["cycle_life"] - vmin) / (vmax - vmin + 1e-9)),
                alpha=0.6, lw=1.2)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_ylim(-0.4, N_CLASSES - 0.6)
    ax.set_title("Ground Truth" if key == "true_classes" else "Predicted", fontsize=12)
    ax.set_xlabel("Window End Cycle"); ax.set_ylabel("RUL Class")
    ax.grid(True, alpha=0.2)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
fig.colorbar(sm, ax=axes, fraction=0.02, pad=0.02).set_label("Cycle Life")
plt.savefig(f"overlay_all_cells_clf_V2_{dataset}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 — Near end-of-life (NEOL) estimation

NEOL = the first window whose predicted class is the most-degraded band.

In [ ]:
LAST_CLASS = N_CLASSES - 1

def neol_from_result(r):
    """First window-end cycle in the most-degraded class, true and predicted.

    The predicted side prefers a genuine (LAST_CLASS-1 -> LAST_CLASS)
    transition and only falls back to the first bare hit, so a single early
    misfire does not decide the estimate on its own.
    """
    cyc, p, t = r["end_cycles"], r["pred_classes"], r["true_classes"]

    hits_t  = cyc[t == LAST_CLASS]
    eol_t   = int(hits_t[0]) if len(hits_t) else r["cycle_life"]

    trans = np.where((p[:-1] == LAST_CLASS - 1) & (p[1:] == LAST_CLASS))[0]
    if len(trans):
        eol_p = int(cyc[trans[0] + 1])
    else:
        hits_p = cyc[p == LAST_CLASS]
        eol_p  = int(hits_p[0]) if len(hits_p) else int(cyc[-1])
    return eol_t, eol_p


cid_list, eol_true_list, eol_pred_list = [], [], []
for cid, r in all_results.items():
    t, p = neol_from_result(r)
    cid_list.append(cid); eol_true_list.append(t); eol_pred_list.append(p)

eol_true = np.array(eol_true_list)
eol_pred = np.array(eol_pred_list)
eol_err  = eol_pred - eol_true
mae_eol  = float(np.mean(np.abs(eol_err)))
mape_eol = float(np.mean(np.abs(eol_err) / (eol_true + 1e-6)) * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Near End-of-Life Estimation", fontsize=13, fontweight="bold")

ax = axes[0]
lim = [min(eol_true.min(), eol_pred.min()) - 50,
       max(eol_true.max(), eol_pred.max()) + 50]
sc = ax.scatter(eol_true, eol_pred, c=np.abs(eol_err), cmap="RdYlGn_r",
                s=60, edgecolors="k", linewidths=0.4, zorder=3)
plt.colorbar(sc, ax=ax, label="|NEOL error| (cycles)")
ax.plot(lim, lim, "k--", lw=1.2, label="Perfect prediction")
ax.fill_between(lim, [l - 100 for l in lim], [l + 100 for l in lim],
                alpha=0.1, color="green", label="+/-100 cycle band")
ax.set_xlabel("True NEOL (cycles)"); ax.set_ylabel("Predicted NEOL (cycles)")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_title("NEOL Parity Plot"); ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
ax.text(0.04, 0.92, f"MAE = {mae_eol:.1f} cycles\nMAPE = {mape_eol:.1f}%",
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))

ax = axes[1]
ax.hist(eol_err, bins=20, color="#1976D2", edgecolor="white", linewidth=0.6)
ax.axvline(0, color="black", lw=1.5, ls="--", label="Zero error")
ax.axvline(eol_err.mean(), color="red", lw=1.5,
           label=f"Mean = {eol_err.mean():.1f} cycles")
ax.set_xlabel("NEOL Prediction Error"); ax.set_ylabel("Count")
ax.set_title("NEOL Error Distribution"); ax.legend(fontsize=9)
ax.grid(True, alpha=0.25); ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f"eol_parity_clf_V2_{dataset}.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Cells evaluated : {len(eol_true)}")
print(f"NEOL MAE        : {mae_eol:.1f} cycles")
print(f"NEOL MAPE       : {mape_eol:.1f}%")
print(f"Mean error      : {eol_err.mean():.1f} cycles  (+ = over-predict)")
print(f"Std error       : {eol_err.std():.1f} cycles")

## 8 — Per-cell summary table

In [ ]:
print(f'{"Cell":>14} | {"CycLife":>7} | {"NEOL_True":>9} | {"NEOL_Pred":>9} | '
      f'{"NEOL_Err":>8} | {"Acc":>7}')
print("-" * 68)
for cid in sorted(all_results):
    r = all_results[cid]
    t, p = neol_from_result(r)
    print(f'{cid:>14} | {r["cycle_life"]:7d} | {t:9d} | {p:9d} | '
          f'{p - t:+8d} | {r["accuracy"]:7.2%}')
print("-" * 68)
print(f'{"MEAN":>14} | {"":>7} | {"":>9} | {"":>9} | {eol_err.mean():+8.1f} | '
      f'{np.mean([r["accuracy"] for r in all_results.values()]):7.2%}')
print(f'{"MAE":>14} | {"":>7} | {"":>9} | {"":>9} | {mae_eol:8.1f} |')
print(f'{"MAPE":>14} | {"":>7} | {"":>9} | {"":>9} | {mape_eol:7.1f}% |')

## 9 — Confusion matrix

One function instead of the three near-identical cells the old notebook
carried (measured 77–88% duplicate text between them).

In [ ]:
def plot_confusion(true, pred, title, normalize=True, ax=None):
    """Plot a labelled confusion matrix.

    Args:
        true: Ground-truth class indices.
        pred: Predicted class indices.
        title: Plot title.
        normalize: Show row-normalised percentages instead of raw counts.
        ax: Existing axis to draw on; a new figure is made when omitted.

    Returns:
        The raw (un-normalised) confusion matrix.
    """
    # labels= is required, not optional: without it sklearn infers the class
    # count from the data present and mislabels (or raises) whenever a split
    # happens to contain fewer than N_CLASSES classes.
    cm = confusion_matrix(true, pred, labels=list(range(N_CLASSES)))
    shown = (cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
             if normalize else cm.astype(float))

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(shown, cmap="Blues", vmin=0, vmax=shown.max())
    ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontsize=11, fontweight="bold")

    thresh = shown.max() / 2
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            txt = f"{shown[i, j]:.1%}\n({cm[i, j]})" if normalize else f"{cm[i, j]}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                    color="white" if shown[i, j] > thresh else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return cm


all_true = np.concatenate([r["true_classes"] for r in all_results.values()])
all_pred = np.concatenate([r["pred_classes"] for r in all_results.values()])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_confusion(all_true, all_pred, "Row-normalised", normalize=True, ax=axes[0])
plot_confusion(all_true, all_pred, "Raw counts", normalize=False, ax=axes[1])
plt.tight_layout()
plt.savefig(f"confusion_matrix_V2_{dataset}.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Overall window accuracy: {(all_true == all_pred).mean():.4f} "
      f"over {len(all_true):,} windows")

off = np.abs(all_true - all_pred)
print(f"Exact class            : {(off == 0).mean():.2%}")
print(f"Within one class       : {(off <= 1).mean():.2%}")